## Introduction<a id='introduction'></a>

**Purpose:**

Collect weather, census, and demand forecasting data. Weather data comes from NOAA, weather.gov, and openweathermap. Census data comes from census.gov.

**Context:**

&emsp; One of the core challenges of an energy provider is to generate and distribute enough energy to meet demand while not overgenerating to accrue loss. Existing forecasting models can struggle to predict demand during extreme weather conditions leading to waste, potential equipment damage, or possibly blackouts. By improving predictions during extreme weather conditions, which can be done by training forecasting models on weather data, energy providers can limit waste, reduce the possibility of equipment damage, and reduce the possibility of blackout. \
&emsp; The purpose of this notebook is to collect the initial data for which to train weather-dependent demand forecasting models.

**Data:**

Census data comes from census.gov (https://data.census.gov/). \
Weather data comes from weather.gov (https://www.weather.gov/) and open-meteo (https://open-meteo.com/en/docs). \
Energy data comes from eia.gov (https://www.eia.gov/opendata/browser/)

**Outcomes from this notebook (Data Wrangling):**

As of 2026-02-27:
- Collected historical weather data
- Collected population data by county
- Collected county latitude and longitude, also ensured latitudes and longitudes could be queried using weather.gov
- Collected forecasted weather data by county
- Collected energy demand, forecasted demand, generation, and interchange. Can also use same api call to get historic data
- Listed cases of severe weather events and detailed impact

## To-Do:

From 2026-02-22:
* Integrate Shubh's code with the rest of notebook
* Write code that collects weather data according to energy grid region. Purpose is to connect energy and weather data
* Choose an event of interest or two from Nayina's document and write code to collect data for event
* Collect detailed information on data quality and reliability.
    * How are data collected?
    * What are the measurement errors?
    * What are errors associated with weather models? What are their biases? What does meterology community think of these models?
* Can we collect inches of precipitation?
* Is it worthwhile to collect data from multiple weather forecasting models?
* Is it worthwhile to collect data from multiple energy forecasting models?

From 2026-02-01:
* Collect historic weather data
    * Troubleshoot OpenWeather or
    * Explore other options
        * **Completed:** Shubh found open-meteo, code at bottom of notebook.
* Explore granularity of energy data, can collect by state, region, but what else?
    * **Completed:** Nayina looked into this, no further granularity, at least of interest, beyond regional. Will want to write code that selects counties associated with each region.
* Identify extreme weather events since 2019
    * Tag if blackouts occurred
    * Document all associated damages, track cost of damages
        * **Completed:** Nayina wrote up a doc with this information. See document under data
* Collect detailed information on data quality and reliability.
    * How are data collected?
    * What are the measurement errors?
    * What are errors associated with weather models? What are their biases? What does meterology community think of these models?
* Can we collect inches of precipitation?
* Is it worthwhile to collect data from multiple weather forecasting models?
* Is it worthwhile to collect data from multiple energy forecasting models?

## Contents<a id='contents'></a>
* [Introduction](#introduction)
* [Contents](#contents)
* [Objectives](#objectives)
* [Collecting Data](#collecting_data)
    * [Census and Geographic Data](#census_geographic_data)
    * [Weather Data](#weather_data)
        * [Weather.gov](#weather_gov)
        * [Open Meteo](#open_meteo)
    * [Energy Data](#energy_data)
        * [EIA](#eia)
* [Sources](#sources)
<!-- * [Cleaning Data](#cleaning_data)
* [Exploring Data](#exploring_data)
    * [Trends by Age](#trends_by_age)
* [Saving Data](#saving_data) -->
<!-- * [Summary](#summary)     -->

## Objectives<a id='objectives'></a>
1. Collect weather data from weather.gov and openweathermap
2. Identify and augment weather data to include at least temperature, atmospheric pressure, humidity, wind speed/direction, precipitation, and cloud cover, ideally to at least the zipcode and hourly level of granularity. Want to be able to collect data for any given day.
3. Standardize weather data
4. Clean and impute weather data
5. Collect census data
6. Standardize census data with weather data
7. Clean and impute census data
8. Collect energy data from eia

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
# Andy's path to project data folder
data_path = '/content/drive/MyDrive/Colab Notebooks/DSCI511/DSCI511 Project/data/'

eia_key = '07U32do8sTra5AjwjbdClxLW4GtAEgO4WEL3FfWA'

## Collecting Data<a id='collecting_data'></a>

### Census and Geographic Data<a id='census_geographic_data'></a>

From census.gov

#### Loading and processing raw data to more suitable format

In [ ]:
# Loading and cleaning county population data
raw_county = pd.read_csv(data_path+'raw/co-est2024-pop.txt')

# cleaning artifacts
county_df = raw_county.iloc[:3144]
county_df['Geographic Area'] = county_df['Geographic Area'].str.slice(1)

# splitting geographic area into 2 columns: county and state
county_state = county_df['Geographic Area'].str.split(',').copy()
county_df['County'] = [x[0] for x in county_state]
county_df['State'] = [x[1].strip() for x in county_state]

# removing Geographic Area now that it is redundant
county_df = county_df.drop(columns=['Geographic Area'])

# changing order so it is more readable
neworder = ['State', 'County', '2020', '2021', '2022', '2023', '2024']
county_df = county_df[neworder]

# saving new dataframe as csv table
county_df.to_csv(data_path+'pop_by_county.csv', index=False)

In [ ]:
# loading code to state name mapper
state_codes = pd.read_csv(data_path+'raw/state_codes.csv')
code_state_mapper = {}
for i in range(len(state_codes)):
    code_state_mapper[state_codes.iloc[i,1]] = state_codes.iloc[i,0]
code_state_mapper['PR'] = 'Puerto Rico'

# Loading and cleaning county latitude and longitude data
raw_county_lat_long = pd.read_csv(data_path+'raw/2025_Gaz_counties_national.txt', delimiter='|')

# dropping unnecessary columns but keeping land area and water area then renaming columns
county_lat_long = raw_county_lat_long.drop(columns=['GEOID', 'GEOIDFQ', 'ANSICODE', 'ALAND_SQMI', 'AWATER_SQMI']).copy()
county_lat_long.rename(columns={'USPS':'State', 'NAME':'County', 'ALAND':'Land Area', 'AWATER':'Water Area',
                                'INTPTLAT':'Latitude', 'INTPTLONG':'Longitude'}, inplace=True)

# converting USPS to State name using mapper
county_lat_long['State'] = list(map(lambda x: code_state_mapper[x], county_lat_long['State']))

# lat and long of St Bernard Parish, LA cant be queried, manually changing
# to lat 29.847650, long -89.697422
my_ind = county_lat_long.index[county_lat_long['County'] == 'St. Bernard Parish'][0]
county_lat_long.loc[my_ind, 'Latitude'] = 29.847650
county_lat_long.loc[my_ind, 'Longitude'] = -89.697422

# saving new dataframe as csv table
county_lat_long.to_csv(data_path+'county_lat_long.csv', index=False)

#### Loading processed county data, both population and latitude and longitude

In [ ]:
county_pop = pd.read_csv(data_path+'pop_by_county.csv')
county_lat_long = pd.read_csv(data_path+'county_lat_long.csv')

In [ ]:
county_pop.head()

,State,County,2020,2021,2022,2023,2024
0,Alabama,Autauga County,"58,909","59,191","59,736","60,436","61,464"
1,Alabama,Baldwin County,"233,244","239,411","246,577","254,107","261,608"
2,Alabama,Barbour County,"24,975","24,517","24,722","24,644","24,358"
3,Alabama,Bibb County,"22,176","22,344","21,983","21,890","22,258"
4,Alabama,Blount County,"59,110","59,050","59,491","59,777","60,163"


In [ ]:
county_lat_long.head()

,State,County,Land Area,Water Area,Latitude,Longitude
0,Alabama,Autauga County,1539631460,25677536,32.532237,-86.646440
1,Alabama,Baldwin County,4117933903,1132678359,30.659218,-87.746067
2,Alabama,Barbour County,2292160152,50523213,31.870253,-85.405103
3,Alabama,Bibb County,1612188713,9572302,33.015893,-87.127148
4,Alabama,Blount County,1670296790,14822589,33.977358,-86.566440


### Weather Data<a id='weather_data'></a>

#### Weather.gov<a id='weather_gov'></a>

For weather forecasts

In [ ]:
def getHourlyForecast(lat: float, long: float) -> pd.DataFrame:
    '''
    Gets hourly forecast data using weather.gov gridpoints forecast hourly api call.
    Inputs:
        lat - latitude in decimal; float
        long - longitude in decimal; float
    Returns:
        forecast_df - hourly weather forecast; pd.DataFrame
    '''
    # api call to points provides api call for hourly forecast so will use that as our primary endpoint
    api='https://api.weather.gov/points/'
    url = api+str(lat)+','+str(long)

    points_response = requests.get(url)

    if points_response.status_code != 200:
        print('ERROR in points request: status code ',points_response.status_code)
        print('latitude: ', lat, '; longitude: ', long)
        return

    hourly_forecast_url = points_response.json()['properties']['forecastHourly']

    forecast_response = requests.get(hourly_forecast_url)

    if forecast_response.status_code != 200:
        print('ERROR in forecast request: status code ', forecast_response.status_code)
        print('latitude: ', lat, '; longitude: ', long)
        return

    forecast_df = pd.DataFrame(forecast_response.json()['properties']['periods'])

    # adding position data
    forecast_df['Latitude'] = lat
    forecast_df['Longitude'] = long

    # converting startTime to date and local time
    forecast_df['Date'] = pd.to_datetime(forecast_df['startTime']).dt.date
    forecast_df['Local Time'] = pd.to_datetime(forecast_df['startTime']).dt.time
    forecast_df['UTC Offset'] = forecast_df['startTime'].str[-6:]

    # unpacking wind speed, prob of precipitation, dewpoint, and rel humidity
    forecast_df['Wind Speed'] = list(map(lambda x: int(x.split()[0]), forecast_df['windSpeed']))
    forecast_df['Wind Speed Unit'] = list(map(lambda x: x.split()[1], forecast_df['windSpeed']))
    forecast_df['Prob Precipitation'] = list(map(lambda x: x['value'], forecast_df['probabilityOfPrecipitation']))
    forecast_df['Dewpoint'] = list(map(lambda x: x['value'], forecast_df['dewpoint']))
    forecast_df['Dewpoint Unit'] = list(map(lambda x: x['unitCode'][-1], forecast_df['dewpoint']))
    forecast_df['Relative Humidity'] = list(map(lambda x: x['value'], forecast_df['relativeHumidity']))

    # dropping unnecessary columns
    drop_cols = ['number', 'name', 'startTime', 'endTime', 'temperatureTrend', 'probabilityOfPrecipitation',
                 'dewpoint', 'relativeHumidity', 'windSpeed', 'icon']
    forecast_df = forecast_df.drop(columns=drop_cols).copy()

    # renaming and reorganizing columns
    rename_cols = {'isDaytime':'Daytime', 'temperature':'Temperature', 'temperatureUnit':'Temperature Unit',
                   'windDirection':'Wind Direction', 'shortForecast':'Short Forecast', 'detailedForecast':'Detailed Forecast'}
    forecast_df.rename(columns=rename_cols, inplace=True)
    neworder = ['Latitude', 'Longitude', 'Date', 'Local Time', 'UTC Offset', 'Daytime', 'Temperature', 'Temperature Unit',
                'Wind Speed', 'Wind Speed Unit', 'Wind Direction', 'Prob Precipitation', 'Dewpoint', 'Dewpoint Unit',
                'Relative Humidity', 'Short Forecast', 'Detailed Forecast']
    forecast_df = forecast_df[neworder]

    return forecast_df

NameError: name 'pd' is not defined

In [ ]:
mask = [x not in ['Puerto Rico', 'Hawaii', 'Alaska'] for x in county_lat_long['State']]
county_lat_long48 = county_lat_long[mask].reset_index()

county_lat_long48

In [ ]:
lat = county_lat_long48['Latitude'][0]
long = county_lat_long48['Longitude'][0]

forecast = getHourlyForecast(lat, long)

for i in range(1, len(county_lat_long48)):
    lat = county_lat_long48['Latitude'][i]
    long = county_lat_long48['Longitude'][i]

    next_forecast = getHourlyForecast(lat, long)

    forecast = pd.concat([forecast, next_forecast], ignore_index=True)

print(forecast.shape)
forecast.to_csv('../data/forecast48.csv', index=False)

del forecast

In [ ]:
forecast = pd.read_csv('../data/forecast48_p2.csv')

In [ ]:
forecast.head()

NameError: name 'forecast' is not defined

#### Open Meteo<a id='open_meteo'></a>

For historical weather

weather code reference table: \
https://www.nodc.noaa.gov/archive/arc0021/0002199/1.1/data/0-data/HTML/WMO-CODE/WMO4677.HTM

In [ ]:
def getHourlyHistorical(lat: float, long: float, start_date: str, end_date: str) -> pd.DataFrame:
    '''
    Gets hourly historical data using open-meteo.com hourly api call.
    Inputs:
        lat - latitude in decimal; float
        long - longitude in decimal; float
        start_date - start date in format yyyy-mm-dd (%Y-%m-%d); str
        end_date - end date in format yyyy-mm-dd (%Y-%m-%d); str
    Returns:
        historical_df - hourly historical weather; pd.DataFrame
    '''
    url = 'https://archive-api.open-meteo.com/v1/archive'
    hourly_params = ','.join(['temperature_2m', 'relativehumidity_2m', 'apparent_temperature',
                              'precipitation', 'rain', 'snowfall', 'weathercode', 'surface_pressure',
                              'windspeed_10m', 'winddirection_10m', 'dew_point_2m'])
    params = {"latitude": lat,
              "longitude": long,
              "start_date": start_date,
              "end_date": end_date,
              "hourly": hourly_params}
    response = requests.get(url, params=params)

    if response.status_code != 200:
        print('ERROR in request: status code ',response.status_code)
        print('latitude: ', lat, '; longitude: ', long)
        print('start_date: ', start_date, '; end_date: ', end_date)
        return

    data = response.json()

    historical_df = pd.DataFrame(data['hourly'])

    # adding lat, long, times, and measurement units
    historical_df['Latitude'] = lat
    historical_df['Longitude'] = long

    my_dt = pd.to_datetime(historical_df['time'], utc=True)
    historical_df['Date'] = my_dt.dt.date
    historical_df['UTC Time'] = my_dt.dt.time

    historical_df['Temperature Unit'] = data['hourly_units']['temperature_2m']
    historical_df['Wind Speed Unit'] = data['hourly_units']['windspeed_10m']
    historical_df['Precipitation Unit'] = data['hourly_units']['precipitation']
    historical_df['Dewpoint Unit'] = data['hourly_units']['dew_point_2m']

    # renaming and reorganizing columns
    rename_cols = {'temperature_2m':'Temperature', 'windspeed_10m':'Wind Speed',
                   'winddirection_10m':'Wind Direction', 'precipitation':'Precipitation',
                   'relativehumidity_2m':'Relative Humidity', 'weathercode':'Weather Code',
                   'dew_point_2m':'Dewpoint'}
    historical_df.rename(columns=rename_cols, inplace=True)

    neworder = ['Latitude', 'Longitude', 'Date', 'UTC Time', 'Temperature', 'Temperature Unit',
                'Wind Speed', 'Wind Speed Unit', 'Wind Direction', 'Precipitation', 'Precipitation Unit',
                'Dewpoint', 'Dewpoint Unit', 'Relative Humidity', 'Weather Code']
    historical_df = historical_df[neworder]

    return historical_df

In [ ]:
lat, long = 39.869437, -77.21775

In [ ]:
df_historical = getHourlyHistorical(lat, long, '2024-01-01', '2024-01-03')

In [ ]:
df_historical.head(2)

,Latitude,Longitude,Date,UTC Time,Temperature,Temperature Unit,Wind Speed,Wind Speed Unit,Wind Direction,Precipitation,Precipitation Unit,Dewpoint,Dewpoint Unit,Relative Humidity,Weather Code
0,39.869437,-77.21775,2024-01-01,00:00:00,4.1,°C,4.6,km/h,315,0.3,mm,-1.5,°C,67,71
1,39.869437,-77.21775,2024-01-01,01:00:00,3.5,°C,3.3,km/h,264,0.0,mm,1.0,°C,84,3


### Energy Data<a id='energy_data'></a>

#### EIA<a id='eia'></a>

Sources: \
https://www.eia.gov/opendata/browser/

In [ ]:
def getDemandForecastRegion(date: str, time: str, region: str, api_key: str, limit=5000) -> pd.DataFrame:
    '''
    Fetches electricity demand, demand forecase, generation, and interchange from eia.gov api
    Region codes available ["CAL", "CAR", "CENT", "FLA", "MIDA", "MIDW", "NE",
                            "NW", "NY", "SE", "SW", "TEN", "TEX", "US48"]
    Inputs:
        date: date in YYYY-MM-DD; str
        time: in HH UTC; str
        region: region code; str
        api_key: api_key for eia.gov, check email; str
    Returns:
        demand_forecast_df: electricity demand, forecast, generation, and interchange; pd.DataFrame
    '''
    url = 'https://api.eia.gov/v2/electricity/rto/region-data/data/?'
    start_time = date+'T'+time
    params = {'frequency':'hourly', 'facets[respondent][]':region, 'data[0]':'value',
          'start':start_time, 'length':limit, 'api_key':api_key}

    demand = requests.get(url, params=params)

    if demand.status_code != 200:
        print('ERROR status code: ', demand.status_code)
        return

    demand_df = pd.DataFrame(demand.json()['response']['data'])

    # standardizing date, time format across datasets
    demand_df['Date'] = pd.to_datetime(demand_df['period']).dt.date
    demand_df['Time UTC'] = pd.to_datetime(demand_df['period']).dt.time

    demand_df['value'] = demand_df['value'].astype(float)

    # renaming and reordering
    demand_df.rename(columns={'respondent':'Region', 'type':'Type',
                              'value':'Value', 'value-units':'Value Units'}, inplace=True)
    cols = ['Date', 'Time UTC', 'Region', 'Type', 'Value', 'Value Units']
    demand_df = demand_df[cols]

    return demand_df

In [ ]:
demand_df = getDemandForecastRegion('2026-01-25', '00', 'US48', eia_key)

demand_df.head()

In [ ]:
datetime_str = demand_df['Date'].astype(str)+'T'+demand_df['Time UTC'].astype(str)
demand_df['Datetime'] = pd.to_datetime(datetime_str, utc=True)

In [ ]:
ax = sns.scatterplot(data=demand_df, x='Datetime', y='Value', hue='Type')

ax.tick_params(axis='x', rotation=45)
handles, _ = ax.get_legend_handles_labels()

ax.legend(handles, ['Demand Forecast', 'Demand', 'Net Generation', 'Total Interchange'])
plt.ylabel('Value (megawatt-hour)')
plt.xlabel('Date');

## Sources<a id='sources'></a>
1. Weather Forecasts (https://www.weather.gov/documentation/services-web-api)
2. Population by County for 2020-2024 (https://www.census.gov/data/tables/time-series/demo/popest/2020s-counties-total.html)
3. County Latitude and Longitude (https://www.census.gov/geographies/reference-files/time-series/geo/gazetteer-files.html)
    * Column meanings (https://www.census.gov/programs-surveys/geography/technical-documentation/records-layout/gaz-record-layouts.html)
4. Energy Demand, Demand Forecast, Generation, and Interchange (https://www.eia.gov/opendata/browser/)
5. Historical Weather (https://open-meteo.com/en/docs)